# I15a — Graph Neural Networks

**COMPSCI 713 — Week 10: AI and Sustainability (GraphCast)**

---

## Overview

Most deep learning assumes data is on a regular grid (images) or a sequence (text). But much of the world is naturally represented as **graphs**: social networks, molecules, road maps, knowledge graphs, and weather systems.

Graph Neural Networks (GNNs) extend deep learning to graph-structured data. Google DeepMind's **GraphCast** (2023) uses GNNs to predict global weather with unprecedented accuracy.

In this lesson you will:
- Understand graph representations (nodes, edges, adjacency)
- Implement message passing from scratch
- Build a Graph Convolutional Network (GCN)
- Apply GNNs to node classification and graph classification
- Understand how GraphCast uses GNNs for weather prediction

### COMPSCI 713 Alignment
- **Week 10 Thursday:** AI and Sustainability: GraphCast (Graph data, GNNs)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from collections import defaultdict

## 1. Graph Representations

A graph G = (V, E) consists of:
- **V:** set of nodes (vertices)
- **E:** set of edges (connections between nodes)
- **Node features X:** a feature vector for each node
- **Edge features:** optional features on edges

**Adjacency matrix A:** A[i][j] = 1 if edge (i,j) exists, 0 otherwise

In [ ]:
# ── Simple graph: Karate Club (Zachary, 1977) ──────────────────────────────
# A classic social network with 34 nodes and 78 edges
# Nodes split into 2 communities (classes 0 and 1)

# Simplified 8-node version for illustration
edges = [
    (0,1),(0,2),(0,3),(1,2),(1,4),(2,5),(3,4),(4,5),(4,6),(5,7),(6,7)
]
n_nodes = 8
labels  = [0, 0, 0, 0, 1, 1, 1, 1]  # community membership

# Build adjacency matrix
A = np.zeros((n_nodes, n_nodes))
for i, j in edges:
    A[i, j] = A[j, i] = 1

# Node features: degree + random features
degrees = A.sum(axis=1, keepdims=True)
np.random.seed(42)
X = np.hstack([degrees, np.random.randn(n_nodes, 3)])

print("Adjacency matrix:")
print(A.astype(int))
print(f"\nNode features shape: {X.shape}")
print(f"Labels: {labels}")

# Visualise
fig, ax = plt.subplots(figsize=(7, 5))
pos = {
    0: (0.2, 0.8), 1: (0.4, 0.9), 2: (0.1, 0.6), 3: (0.3, 0.6),
    4: (0.6, 0.6), 5: (0.8, 0.6), 6: (0.7, 0.4), 7: (0.9, 0.4)
}
colors = ['steelblue' if l == 0 else 'coral' for l in labels]
for i, j in edges:
    x_vals = [pos[i][0], pos[j][0]]
    y_vals = [pos[i][1], pos[j][1]]
    ax.plot(x_vals, y_vals, 'k-', alpha=0.4, linewidth=1.5)
for node, (x, y) in pos.items():
    ax.scatter(x, y, s=400, c=colors[node], zorder=5, edgecolors='black')
    ax.text(x, y, str(node), ha='center', va='center', fontsize=11, fontweight='bold')
ax.set_title('Graph: 2 Communities (blue vs red)')
ax.axis('off')
plt.tight_layout()
plt.show()

## 2. Message Passing — The Core of GNNs

The key idea in GNNs: each node **aggregates information from its neighbours** and updates its own representation.

$$h_v^{(k)} = \text{UPDATE}\left(h_v^{(k-1)},\ \text{AGGREGATE}\left(\{h_u^{(k-1)} : u \in \mathcal{N}(v)\}\right)\right)$$

After K layers, each node's representation captures information from its K-hop neighbourhood.

In [ ]:
def message_passing(A, H, W):
    """
    One layer of Graph Convolution (GCN).
    H_new = ReLU(D^{-1/2} A_hat D^{-1/2} H W)
    where A_hat = A + I (self-loops)
    """
    n = A.shape[0]
    A_hat = A + np.eye(n)  # add self-loops
    D = np.diag(A_hat.sum(axis=1))
    D_inv_sqrt = np.diag(1.0 / np.sqrt(np.diag(D)))
    A_norm = D_inv_sqrt @ A_hat @ D_inv_sqrt  # symmetric normalisation
    H_new = A_norm @ H @ W
    return np.maximum(0, H_new)  # ReLU


# Manually run 2 layers of message passing
np.random.seed(0)
n_features = X.shape[1]
hidden_dim = 8
out_dim    = 2

W1 = np.random.randn(n_features, hidden_dim) * 0.1
W2 = np.random.randn(hidden_dim, out_dim) * 0.1

H0 = X
H1 = message_passing(A, H0, W1)
H2 = message_passing(A, H1, W2)

print(f"Layer 0 (input):  {H0.shape}")
print(f"Layer 1 (hidden): {H1.shape}")
print(f"Layer 2 (output): {H2.shape}")
print()
print("Node embeddings after 2 layers:")
for i, (emb, label) in enumerate(zip(H2, labels)):
    print(f"  Node {i} (class {label}): [{emb[0]:+.3f}, {emb[1]:+.3f}]")

In [ ]:
# Visualise node embeddings in 2D
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, H, title in [(axes[0], H1, 'After Layer 1'), (axes[1], H2, 'After Layer 2')]:
    # Use first 2 dims for visualisation
    colors = ['steelblue' if l == 0 else 'coral' for l in labels]
    ax.scatter(H[:, 0], H[:, 1], c=colors, s=200, edgecolors='black', zorder=5)
    for i, (x, y) in enumerate(H[:, :2]):
        ax.annotate(str(i), (x, y), textcoords='offset points',
                    xytext=(5, 5), fontsize=10)
    ax.set_title(f'Node Embeddings {title}')
    ax.set_xlabel('Dim 0')
    ax.set_ylabel('Dim 1')
    ax.grid(True, alpha=0.3)

plt.suptitle('GCN: Nodes of same community cluster together', fontsize=12)
plt.tight_layout()
plt.show()

## 3. GNN with PyTorch (if available)

In [ ]:
try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F

    class GCNLayer(nn.Module):
        def __init__(self, in_features, out_features):
            super().__init__()
            self.linear = nn.Linear(in_features, out_features, bias=False)

        def forward(self, H, A_norm):
            return F.relu(A_norm @ self.linear(H))

    class GCN(nn.Module):
        def __init__(self, n_features, hidden, n_classes):
            super().__init__()
            self.layer1 = GCNLayer(n_features, hidden)
            self.layer2 = GCNLayer(hidden, n_classes)

        def forward(self, H, A_norm):
            H = self.layer1(H, A_norm)
            H = self.layer2(H, A_norm)
            return F.log_softmax(H, dim=1)

    # Prepare tensors
    n = A.shape[0]
    A_hat = A + np.eye(n)
    D_inv_sqrt = np.diag(1.0 / np.sqrt(A_hat.sum(axis=1)))
    A_norm_np = D_inv_sqrt @ A_hat @ D_inv_sqrt

    X_t     = torch.FloatTensor(X)
    A_t     = torch.FloatTensor(A_norm_np)
    y_t     = torch.LongTensor(labels)

    model     = GCN(X.shape[1], 16, 2)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

    losses = []
    for epoch in range(200):
        model.train()
        optimizer.zero_grad()
        out  = model(X_t, A_t)
        loss = F.nll_loss(out, y_t)
        loss.backward()
        optimizer.step()
        losses.append(loss.item())

    model.eval()
    with torch.no_grad():
        preds = model(X_t, A_t).argmax(dim=1)
    acc = (preds == y_t).float().mean().item()
    print(f"GCN (PyTorch) node classification accuracy: {acc:.2%}")

    plt.figure(figsize=(8, 3))
    plt.plot(losses, color='steelblue')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title('GCN Training Loss')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

except ImportError:
    print("PyTorch not available. The numpy implementation above demonstrates the same concepts.")

## 4. GraphCast — GNNs for Weather Prediction

Google DeepMind's **GraphCast** (Lam et al., 2023) represents the Earth as a graph:
- **Nodes:** grid points on the Earth's surface (~40,000 nodes)
- **Edges:** connections between nearby grid points
- **Node features:** temperature, wind speed, pressure, humidity at each point
- **Task:** predict weather 10 days ahead

**Architecture:**
1. **Encoder:** map grid data to graph node features
2. **Processor:** 16 rounds of message passing (GNN layers)
3. **Decoder:** map graph features back to grid predictions

**Why GNNs?**
- Weather is inherently a graph problem: each location is influenced by its neighbours
- GNNs respect the **inductive bias** of the problem (locality, permutation equivariance)
- More accurate than traditional numerical weather prediction at 10-day forecasts

In [ ]:
# ── Mini weather graph simulation ──────────────────────────────────────────
# Simulate a 5x5 grid of weather stations

grid_size = 5
n_stations = grid_size ** 2

# Build grid graph (each station connected to 4 neighbours)
A_weather = np.zeros((n_stations, n_stations))
for r in range(grid_size):
    for c in range(grid_size):
        idx = r * grid_size + c
        for dr, dc in [(-1,0),(1,0),(0,-1),(0,1)]:
            nr, nc = r+dr, c+dc
            if 0 <= nr < grid_size and 0 <= nc < grid_size:
                A_weather[idx, nr*grid_size+nc] = 1

# Simulate temperature field (warm in centre, cold at edges)
np.random.seed(42)
temps = np.array([
    15 - 3*abs(r - grid_size//2) - 3*abs(c - grid_size//2) + np.random.randn()
    for r in range(grid_size) for c in range(grid_size)
])

# One step of GNN message passing = spatial smoothing
A_hat = A_weather + np.eye(n_stations)
D_inv = np.diag(1.0 / A_hat.sum(axis=1))
A_norm_w = D_inv @ A_hat

temps_smoothed = A_norm_w @ temps  # one message passing step

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, data, title in [
    (axes[0], temps.reshape(grid_size, grid_size), 'Original Temperature'),
    (axes[1], temps_smoothed.reshape(grid_size, grid_size), 'After 1 GNN Layer (smoothed)')
]:
    im = ax.imshow(data, cmap='RdYlBu_r', vmin=temps.min(), vmax=temps.max())
    plt.colorbar(im, ax=ax, label='°C')
    ax.set_title(title)
    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude')

plt.suptitle('GNN Message Passing on Weather Grid (GraphCast concept)', fontsize=12)
plt.tight_layout()
plt.show()

## 5. GNN Variants

| Model | Aggregation | Key Feature |
|---|---|---|
| **GCN** (Kipf & Welling, 2017) | Normalised sum | Simple, effective for node classification |
| **GraphSAGE** (Hamilton et al., 2017) | Sample + aggregate | Inductive (works on unseen nodes) |
| **GAT** (Veličković et al., 2018) | Attention-weighted | Learns which neighbours matter more |
| **GIN** (Xu et al., 2019) | Sum + MLP | Most expressive (as powerful as WL test) |
| **GraphCast** (Lam et al., 2023) | Message passing | Weather prediction at global scale |

## 6. Summary

### Key Takeaways
- Graphs represent relational data: nodes (entities) + edges (relationships)
- **Message passing** is the core operation: aggregate neighbour features, update node representation
- **GCN** normalises the adjacency matrix to prevent scale issues
- After K layers, each node captures its K-hop neighbourhood
- **GraphCast** uses GNNs to predict global weather — outperforming traditional numerical methods
- GNNs are used in drug discovery, social networks, recommendation systems, and physics simulation

### Next Steps
- **E09** — Self-Supervised Learning (pre-training GNNs without labels)
- **E14** — Efficient and Green AI (reducing the cost of large GNN training)